# LLM-as-Judge End-to-End Workflow (Customer Support QA)

This notebook implements a complete LLM-as-judge evaluation pipeline with:
- Live API execution by default.
- `gpt-4o` as the default candidate and judge model.
- Optional Anthropic and Gemini provider adapters.
- Pointwise rubric scoring and pairwise head-to-head judging.
- Full artifact export and visual analytics.

## Research Primer

**LLM-as-judge** uses a strong model to evaluate model outputs using a structured rubric or pairwise comparison. It is useful for subjective dimensions (helpfulness, clarity, policy compliance) where exact-match metrics are weak.

Known failure modes and mitigations used in this notebook:
- **Position bias**: Run pairwise comparisons with response order swap and aggregate both passes.
- **Verbosity bias**: Explicitly reward concise, relevant answers in rubric.
- **Self-enhancement bias**: Judge sees anonymized responses (`A`/`B`), not provider names.
- **Prompt sensitivity**: Use fixed, versioned grading prompts and strict JSON schema.

Why combine methods:
- **Pointwise** gives fine-grained diagnostics per dimension.
- **Pairwise** provides robust ranking signals across competing models.

Primary references:
1. MT-Bench / Chatbot Arena: https://arxiv.org/abs/2306.05685
2. G-Eval: https://arxiv.org/abs/2303.16634
3. Self-Refine-style feedback loops: https://arxiv.org/abs/2305.17926
4. OpenAI eval design guide: https://platform.openai.com/docs/guides/evals-design
5. OpenAI graders guide: https://platform.openai.com/docs/guides/graders

## 1) Environment Setup

Run the next cell once to install dependencies in your notebook environment.

In [ ]:
# Uncomment to install packages if needed.
# %pip install -q openai anthropic google-genai pandas numpy scipy matplotlib seaborn python-dotenv tqdm

In [ ]:
from __future__ import annotations

import itertools
import json
import math
import os
import random
import re
import time
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Protocol, Tuple

import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns
from dotenv import load_dotenv
from matplotlib import pyplot as plt
from tqdm.auto import tqdm

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

try:
    import anthropic
except ImportError:
    anthropic = None

try:
    from google import genai as google_genai
except ImportError:
    google_genai = None

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 140)
load_dotenv()

## 2) Configuration

In [ ]:
CONFIG: Dict[str, Any] = {
    'seed': 42,
    'dataset_size': 30,
    'live_mode': True,
    'max_retries': 3,
    'retry_base_delay_sec': 1.2,
    'judge_repeat_fraction': 0.15,
    'primary_metric': 'composite_score',
    'guardrail_metric': 'safety_violation_rate',
    'ship_threshold_composite': 4.0,
    'iterate_threshold_composite': 3.4,
    'ship_max_safety_violation_rate': 0.05,
    'iterate_max_safety_violation_rate': 0.12,
    'providers': {
        'openai': {
            'enabled': True,
            'api_key_env': 'OPENAI_API_KEY',
            'candidate_model': os.getenv('OPENAI_CANDIDATE_MODEL', 'gpt-4o'),
            'judge_model': os.getenv('OPENAI_JUDGE_MODEL', 'gpt-4o'),
        },
        'anthropic': {
            'enabled': True,
            'api_key_env': 'ANTHROPIC_API_KEY',
            'candidate_model': os.getenv('ANTHROPIC_MODEL'),
        },
        'gemini': {
            'enabled': True,
            'api_key_env': 'GEMINI_API_KEY',
            'candidate_model': os.getenv('GEMINI_MODEL'),
        },
    },
}

RUBRIC_DIMENSIONS = ['helpfulness', 'factuality', 'policy_safety', 'clarity', 'conciseness']
RUBRIC_WEIGHTS = {
    'helpfulness': 0.30,
    'factuality': 0.30,
    'policy_safety': 0.20,
    'clarity': 0.10,
    'conciseness': 0.10,
}

assert abs(sum(RUBRIC_WEIGHTS.values()) - 1.0) < 1e-9

run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
ARTIFACT_DIR = Path('artifacts') / 'llm_judge_runs' / run_id
PLOTS_DIR = ARTIFACT_DIR / 'plots'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print('Run ID:', run_id)
print('Artifacts:', ARTIFACT_DIR)
print('Live mode:', CONFIG['live_mode'])

## 3) Public Interfaces / Types

In [ ]:
@dataclass
class EvalExample:
    example_id: str
    category: str
    difficulty: str
    user_query: str
    reference_answer: str
    policy_tags: List[str]


@dataclass
class ModelResponse:
    example_id: str
    model_alias: str
    provider: str
    model_id: str
    response_text: str
    latency_ms: float
    error: Optional[str]


@dataclass
class PointwiseJudgeResult:
    example_id: str
    model_alias: str
    scores: Dict[str, int]
    composite_score: float
    safety_violation: bool
    rationale: str


@dataclass
class PairwiseJudgeResult:
    example_id: str
    model_a: str
    model_b: str
    winner: str  # model_a | model_b | tie
    confidence: float
    order_used: str

## 4) Sample Evaluation Data (Generated + Saved)

In [ ]:
CATEGORY_SCENARIOS = {
  "billing": [
    {
      "difficulty": "easy",
      "user_query": "I was charged for an add-on I thought I canceled last month. Can you explain why?",
      "reference_answer": "I can help check that. Please confirm the last 4 digits of the payment method and the invoice number so I can review your billing timeline. If the add-on cancellation was submitted after the invoice cutoff, the current charge may still apply and the cancellation takes effect next cycle. If we confirm an incorrect charge, I can process a correction or refund according to policy.",
      "policy_tags": [
"verification_required",
"billing_timeline",
"refund_if_billing_error"
      ]
    },
    {
      "difficulty": "medium",
      "user_query": "Why is my first bill after upgrading higher than expected?",
      "reference_answer": "That is usually due to proration. When a plan changes mid-cycle, we charge the difference for the remaining days of the current cycle, then your new standard rate starts at renewal. I can share a line-by-line breakdown from your invoice so each prorated item is clear.",
      "policy_tags": [
"proration",
"billing_explanation",
"transparent_breakdown"
      ]
    },
    {
      "difficulty": "medium",
      "user_query": "I see two identical charges today. Did I get billed twice?",
      "reference_answer": "Thanks for flagging this. One entry may be a temporary authorization hold while the other is the settled transaction. If both settle as posted charges, we will open a duplicate-charge review and issue a refund if confirmed. Please share the charge timestamps and payment method so we can verify quickly.",
      "policy_tags": [
"duplicate_charge_review",
"auth_hold_vs_settled",
"refund_eligible"
      ]
    },
    {
      "difficulty": "hard",
      "user_query": "Autopay failed and I got a late fee, but my card had funds. Can you reverse it?",
      "reference_answer": "I can investigate and request a fee reversal if this was a processor or system issue. Please confirm your account email and the payment attempt time. If logs show a platform-side failure or first-time exception criteria are met, we can waive the late fee and help update your payment method to prevent repeats.",
      "policy_tags": [
"late_fee_review",
"payment_failure_investigation",
"one_time_waiver_policy"
      ]
    },
    {
      "difficulty": "easy",
      "user_query": "Can I get a downloadable invoice for last month for reimbursement?",
      "reference_answer": "Yes. Go to Billing > Invoices, select last month, and choose Download PDF. If you need your company address or tax ID on the invoice, share the exact details and I can update your billing profile before you re-download.",
      "policy_tags": [
"invoice_download",
"billing_profile_update",
"tax_fields"
      ]
    }
  ],
  "refunds": [
    {
      "difficulty": "easy",
      "user_query": "I accidentally renewed the annual plan two days ago. Can I still get a refund?",
      "reference_answer": "You may be eligible if the request is within the refund window. I can submit it right away after account verification. If approved, the refund returns to the original payment method, typically within 5-10 business days depending on your bank.",
      "policy_tags": [
"refund_window",
"verification_required",
"bank_settlement_timeline"
      ]
    },
    {
      "difficulty": "medium",
      "user_query": "I returned one item from my order. Why did I only get a partial refund?",
      "reference_answer": "Partial refunds usually reflect item-level pricing, discounts, or non-refundable fees like expedited shipping. I can provide the exact calculation and confirm whether the returned item met full-refund conditions. If something is off, we will correct the amount.",
      "policy_tags": [
"partial_refund_math",
"non_refundable_fees",
"refund_correction"
      ]
    },
    {
      "difficulty": "hard",
      "user_query": "My refund was issued to a card that is now closed. What happens now?",
      "reference_answer": "Most banks reroute refunds from closed cards to the replacement account automatically. If that fails, the bank rejects the refund and it returns to us, after which we can issue an alternate payout method where policy allows. I can start a refund trace and share status updates.",
      "policy_tags": [
"refund_trace",
"closed_card_refund",
"alternate_payout_policy"
      ]
    },
    {
      "difficulty": "medium",
      "user_query": "I was charged after canceling my trial. Please refund immediately.",
      "reference_answer": "I can review the cancellation timestamp and billing event. If the cancellation completed before trial end, we will reverse the charge. If cancellation happened after conversion, we can check if a courtesy exception applies under policy.",
      "policy_tags": [
"trial_conversion_timing",
"billing_dispute",
"courtesy_exception_policy"
      ]
    },
    {
      "difficulty": "easy",
      "user_query": "How long does it take for a refund to show up after approval?",
      "reference_answer": "Once approved, refunds typically post in 5-10 business days. Debit cards may appear faster, while international cards can take longer. I can confirm your refund reference ID so your bank can trace it if needed.",
      "policy_tags": [
"refund_eta",
"reference_id",
"bank_dependency"
      ]
    }
  ],
  "shipping": [
    {
      "difficulty": "medium",
      "user_query": "My tracking has been stuck on 'Label created' for 4 days. Is it lost?",
      "reference_answer": "That status means the carrier has not scanned the package yet. If no movement occurs after the carrier threshold, we can open a lost-package investigation and provide replacement or refund options according to policy. I can file that now if you share the order number.",
      "policy_tags": [
"carrier_scan_delay",
"lost_package_threshold",
"replacement_or_refund"
      ]
    },
    {
      "difficulty": "easy",
      "user_query": "I entered the wrong shipping address. Can you change it now?",
      "reference_answer": "If the order has not entered carrier handoff, we can still update the address. Please send the correct address in full, and I will check fulfillment status immediately. If already shipped, I can help with carrier intercept options where available.",
      "policy_tags": [
"address_change_window",
"fulfillment_status_check",
"carrier_intercept"
      ]
    },
    {
      "difficulty": "hard",
      "user_query": "I paid for expedited shipping but it arrived late. Can I get shipping charges back?",
      "reference_answer": "You may qualify for expedited-fee reimbursement when delivery misses the guaranteed window and no exclusion applies (for example weather or carrier exception). I can verify the promise date and submit a shipping-credit request right away.",
      "policy_tags": [
"delivery_guarantee",
"shipping_fee_refund",
"exclusion_policy"
      ]
    },
    {
      "difficulty": "medium",
      "user_query": "Tracking says delivered, but I never received the package.",
      "reference_answer": "I am sorry you are dealing with that. First, please confirm your delivery address and check nearby safe-drop areas. If still missing, we can start a carrier proof-of-delivery case and then process replacement or refund based on investigation results.",
      "policy_tags": [
"proof_of_delivery_case",
"porch_piracy_process",
"replacement_or_refund_policy"
      ]
    },
    {
      "difficulty": "easy",
      "user_query": "Do I have to pay customs fees for an international shipment?",
      "reference_answer": "Customs duties depend on destination country rules and product category. At checkout we show estimated duties when prepaid options are available. If duties are unpaid at checkout, the carrier may collect them at delivery.",
      "policy_tags": [
"international_duties",
"checkout_disclosure",
"carrier_collection"
      ]
    }
  ],
  "account_access": [
    {
      "difficulty": "easy",
      "user_query": "I forgot my password and reset emails never arrive.",
      "reference_answer": "Please check spam and promotions folders, then whitelist our sender domain. If nothing arrives in 10 minutes, I can trigger a secure reset manually after verifying your account details. For security, never share your password in chat.",
      "policy_tags": [
"secure_reset",
"email_deliverability",
"no_password_collection"
      ]
    },
    {
      "difficulty": "hard",
      "user_query": "I lost my phone and cannot pass 2FA. I am locked out.",
      "reference_answer": "I can help with account recovery. We will verify identity with approved methods (for example recent billing details and account metadata), then reset 2FA and require a new authenticator setup at next login. Recovery requests may require a short review period for security.",
      "policy_tags": [
"identity_verification",
"2fa_recovery",
"security_review"
      ]
    },
    {
      "difficulty": "medium",
      "user_query": "My account says locked after too many attempts. How long until it unlocks?",
      "reference_answer": "Temporary lockouts usually clear automatically after the cooldown period shown on the login page. If urgent, I can verify ownership and send a secure unlock link. Repeated failed attempts can extend lock time for security reasons.",
      "policy_tags": [
"lockout_cooldown",
"secure_unlock",
"abuse_prevention"
      ]
    },
    {
      "difficulty": "medium",
      "user_query": "I got a login alert from another country. What should I do?",
      "reference_answer": "Treat this as potentially unauthorized. Immediately reset your password, review active sessions, and revoke unknown devices. I can force sign-out on all sessions and enable stronger security settings on your account.",
      "policy_tags": [
"account_takeover_response",
"session_revocation",
"security_hardening"
      ]
    },
    {
      "difficulty": "hard",
      "user_query": "Can you merge my old account with my current one so I keep my purchase history?",
      "reference_answer": "We can review merge eligibility after verifying ownership of both accounts. Some data can be transferred (purchase history, entitlements), while certain security and compliance records may remain separate. I will open a merge request and share expected turnaround.",
      "policy_tags": [
"dual_account_verification",
"data_merge_policy",
"compliance_limits"
      ]
    }
  ],
  "technical_troubleshooting": [
    {
      "difficulty": "medium",
      "user_query": "The mobile app crashes right after launch since the last update.",
      "reference_answer": "Let us isolate the issue quickly. Please confirm app version, device model, and OS version. As immediate steps, force close, clear cache, and reinstall. If crash persists, I can collect diagnostic logs and escalate to engineering with a priority bug ticket.",
      "policy_tags": [
"triage_steps",
"device_context_required",
"engineering_escalation"
      ]
    },
    {
      "difficulty": "hard",
      "user_query": "My data syncs on web but not on iOS. How can I fix cross-device sync?",
      "reference_answer": "This can happen if background refresh, account scope, or token state differs across devices. Please confirm both devices use the same account, then re-authenticate on iOS and trigger a manual sync. If mismatch remains, I can run a backend sync health check.",
      "policy_tags": [
"cross_device_sync",
"reauthenticate",
"backend_health_check"
      ]
    },
    {
      "difficulty": "easy",
      "user_query": "Why can't I upload files larger than 20MB?",
      "reference_answer": "The current upload limit is 20MB per file on your plan tier. You can compress files, split them, or upgrade to a tier with larger limits. I can share exact limits by plan so you can choose the best option.",
      "policy_tags": [
"plan_limits",
"workarounds",
"upgrade_path"
      ]
    },
    {
      "difficulty": "medium",
      "user_query": "I stopped receiving notifications even though settings are on.",
      "reference_answer": "Please verify system-level notification permissions and battery optimization settings, since these can override in-app preferences. Then send a test notification from settings. If it still fails, I can check delivery logs for your account.",
      "policy_tags": [
"permission_check",
"device_overrides",
"notification_logs"
      ]
    },
    {
      "difficulty": "hard",
      "user_query": "Our integration keeps hitting 429 errors. What should we change?",
      "reference_answer": "HTTP 429 means rate limits are being exceeded. Implement exponential backoff with jitter, respect retry-after headers, and smooth burst traffic with queueing. If sustained throughput needs exceed current limits, we can discuss a higher API quota plan.",
      "policy_tags": [
"rate_limit_handling",
"retry_backoff",
"quota_management"
      ]
    }
  ],
  "subscriptions": [
    {
      "difficulty": "medium",
      "user_query": "If I downgrade now, do I lose features immediately or at renewal?",
      "reference_answer": "Downgrades usually take effect at the next renewal so you keep current features for the paid period. I can confirm your exact effective date and what feature set will change afterward.",
      "policy_tags": [
"downgrade_effective_date",
"feature_entitlement_window",
"renewal_policy"
      ]
    },
    {
      "difficulty": "easy",
      "user_query": "What is the difference between pausing and canceling my subscription?",
      "reference_answer": "Pausing temporarily suspends billing and access rules for a defined period, while canceling ends renewal permanently. Depending on plan and region, pause may not be available. I can check what options your account supports.",
      "policy_tags": [
"pause_vs_cancel",
"regional_policy_variance",
"account_eligibility"
      ]
    },
    {
      "difficulty": "medium",
      "user_query": "How do I add one more seat to our team plan?",
      "reference_answer": "Go to Team Settings > Billing > Manage Seats and increase seat count by one. Charges are typically prorated for the remaining cycle days. I can also apply this change for you if you prefer.",
      "policy_tags": [
"seat_management",
"proration",
"admin_action_path"
      ]
    },
    {
      "difficulty": "hard",
      "user_query": "Can I move my renewal date to align with our fiscal month-end?",
      "reference_answer": "We can review billing-cycle alignment options. In many cases this requires a one-time prorated adjustment or contract amendment depending on plan type. I can escalate to billing operations to propose the cleanest transition date.",
      "policy_tags": [
"billing_cycle_alignment",
"contract_change",
"billing_ops_escalation"
      ]
    },
    {
      "difficulty": "easy",
      "user_query": "Do you offer a student discount and how do I verify eligibility?",
      "reference_answer": "Student discounts may be available in supported regions through verified academic status. You typically complete verification with a school email or approved partner flow. I can share the exact steps and eligibility page for your country.",
      "policy_tags": [
"student_discount_policy",
"eligibility_verification",
"region_specific_terms"
      ]
    }
  ]
}


def build_eval_examples(seed: int = 42, dataset_size: int = 30) -> List[EvalExample]:
    rows: List[Dict[str, Any]] = []
    for category, scenarios in CATEGORY_SCENARIOS.items():
        for s in scenarios:
            rows.append(
                {
                    'category': category,
                    'difficulty': s['difficulty'],
                    'user_query': s['user_query'],
                    'reference_answer': s['reference_answer'],
                    'policy_tags': s['policy_tags'],
                }
            )

    rng = random.Random(seed)
    rng.shuffle(rows)

    if dataset_size <= len(rows):
        rows = rows[:dataset_size]
    else:
        extended: List[Dict[str, Any]] = []
        while len(extended) < dataset_size:
            extended.extend(rows)
        rows = extended[:dataset_size]

    examples: List[EvalExample] = []
    for idx, row in enumerate(rows, start=1):
        examples.append(
            EvalExample(
                example_id=f'CS{idx:03d}',
                category=row['category'],
                difficulty=row['difficulty'],
                user_query=row['user_query'],
                reference_answer=row['reference_answer'],
                policy_tags=row['policy_tags'],
            )
        )
    return examples


def save_jsonl(records: List[Dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=True) + '\n')


examples = build_eval_examples(seed=CONFIG['seed'], dataset_size=CONFIG['dataset_size'])
dataset_records = [asdict(x) for x in examples]

DATASET_PATH = Path('data') / 'customer_support_eval_samples.jsonl'
save_jsonl(dataset_records, DATASET_PATH)
print(f'Saved {len(dataset_records)} examples -> {DATASET_PATH}')

dataset_df = pd.DataFrame(dataset_records)
display(dataset_df.head(8))
dataset_df['category'].value_counts().to_frame('count')

## 5) Provider Adapters (OpenAI required path; Anthropic/Gemini optional)

In [ ]:
class TextModelAdapter(Protocol):
    provider: str
    model_id: str
    model_alias: str

    def generate(
        self,
        prompt: str,
        system_prompt: str = '',
        temperature: float = 0.2,
        max_tokens: int = 600,
    ) -> str:
        ...


@dataclass
class OpenAIAdapter:
    model_id: str
    model_alias: str
    api_key: str
    provider: str = 'openai'

    def __post_init__(self) -> None:
        if OpenAI is None:
            raise ImportError('openai package is not installed')
        self.client = OpenAI(api_key=self.api_key)

    def generate(
        self,
        prompt: str,
        system_prompt: str = '',
        temperature: float = 0.2,
        max_tokens: int = 600,
    ) -> str:
        messages = []
        if system_prompt:
            messages.append({'role': 'system', 'content': system_prompt})
        messages.append({'role': 'user', 'content': prompt})

        kwargs = {
            'model': self.model_id,
            'messages': messages,
            'temperature': temperature,
            'max_tokens': max_tokens,
        }
        try:
            completion = self.client.chat.completions.create(**kwargs)
        except TypeError:
            kwargs.pop('max_tokens', None)
            kwargs['max_completion_tokens'] = max_tokens
            completion = self.client.chat.completions.create(**kwargs)

        text = completion.choices[0].message.content
        if isinstance(text, list):
            return ''.join(str(x) for x in text).strip()
        return (text or '').strip()


@dataclass
class AnthropicAdapter:
    model_id: str
    model_alias: str
    api_key: str
    provider: str = 'anthropic'

    def __post_init__(self) -> None:
        if anthropic is None:
            raise ImportError('anthropic package is not installed')
        self.client = anthropic.Anthropic(api_key=self.api_key)

    def generate(
        self,
        prompt: str,
        system_prompt: str = '',
        temperature: float = 0.2,
        max_tokens: int = 600,
    ) -> str:
        message = self.client.messages.create(
            model=self.model_id,
            system=system_prompt if system_prompt else None,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=temperature,
            max_tokens=max_tokens,
        )
        parts = []
        for block in message.content:
            if getattr(block, 'type', None) == 'text':
                parts.append(block.text)
        return ''.join(parts).strip()


@dataclass
class GeminiAdapter:
    model_id: str
    model_alias: str
    api_key: str
    provider: str = 'gemini'

    def __post_init__(self) -> None:
        if google_genai is None:
            raise ImportError('google-genai package is not installed')
        self.client = google_genai.Client(api_key=self.api_key)

    def generate(
        self,
        prompt: str,
        system_prompt: str = '',
        temperature: float = 0.2,
        max_tokens: int = 600,
    ) -> str:
        merged_prompt = prompt if not system_prompt else f"System instructions:\n{system_prompt}\n\nUser request:\n{prompt}"
        try:
            response = self.client.models.generate_content(
                model=self.model_id,
                contents=merged_prompt,
                config={'temperature': temperature, 'max_output_tokens': max_tokens},
            )
        except Exception:
            response = self.client.models.generate_content(model=self.model_id, contents=merged_prompt)

        text = getattr(response, 'text', None)
        if text:
            return text.strip()

        parts = []
        for candidate in getattr(response, 'candidates', []) or []:
            content = getattr(candidate, 'content', None)
            for part in getattr(content, 'parts', []) or []:
                if hasattr(part, 'text') and part.text:
                    parts.append(part.text)
        return ''.join(parts).strip()


def provider_status_table(config: Dict[str, Any]) -> pd.DataFrame:
    rows = []
    pkg_openai = OpenAI is not None
    pkg_anthropic = anthropic is not None
    pkg_gemini = google_genai is not None

    rows.append(
        {
            'provider': 'openai',
            'package_installed': pkg_openai,
            'api_key_present': bool(os.getenv(config['providers']['openai']['api_key_env'])),
            'model_configured': bool(config['providers']['openai']['candidate_model']),
            'candidate_model': config['providers']['openai']['candidate_model'],
        }
    )
    rows.append(
        {
            'provider': 'anthropic',
            'package_installed': pkg_anthropic,
            'api_key_present': bool(os.getenv(config['providers']['anthropic']['api_key_env'])),
            'model_configured': bool(config['providers']['anthropic']['candidate_model']),
            'candidate_model': config['providers']['anthropic']['candidate_model'],
        }
    )
    rows.append(
        {
            'provider': 'gemini',
            'package_installed': pkg_gemini,
            'api_key_present': bool(os.getenv(config['providers']['gemini']['api_key_env'])),
            'model_configured': bool(config['providers']['gemini']['candidate_model']),
            'candidate_model': config['providers']['gemini']['candidate_model'],
        }
    )
    return pd.DataFrame(rows)


def build_candidate_adapters(config: Dict[str, Any]) -> List[TextModelAdapter]:
    adapters: List[TextModelAdapter] = []

    openai_cfg = config['providers']['openai']
    openai_key = os.getenv(openai_cfg['api_key_env'])
    if openai_cfg['enabled'] and openai_cfg['candidate_model'] and openai_key and OpenAI is not None:
        adapters.append(
            OpenAIAdapter(
                model_id=openai_cfg['candidate_model'],
                model_alias=f"openai:{openai_cfg['candidate_model']}",
                api_key=openai_key,
            )
        )

    anthropic_cfg = config['providers']['anthropic']
    anthropic_key = os.getenv(anthropic_cfg['api_key_env'])
    if (
        anthropic_cfg['enabled']
        and anthropic_cfg['candidate_model']
        and anthropic_key
        and anthropic is not None
    ):
        adapters.append(
            AnthropicAdapter(
                model_id=anthropic_cfg['candidate_model'],
                model_alias=f"anthropic:{anthropic_cfg['candidate_model']}",
                api_key=anthropic_key,
            )
        )

    gemini_cfg = config['providers']['gemini']
    gemini_key = os.getenv(gemini_cfg['api_key_env'])
    if gemini_cfg['enabled'] and gemini_cfg['candidate_model'] and gemini_key and google_genai is not None:
        adapters.append(
            GeminiAdapter(
                model_id=gemini_cfg['candidate_model'],
                model_alias=f"gemini:{gemini_cfg['candidate_model']}",
                api_key=gemini_key,
            )
        )

    return adapters


def build_judge_adapter(config: Dict[str, Any], candidates: List[TextModelAdapter]) -> TextModelAdapter:
    openai_cfg = config['providers']['openai']
    openai_key = os.getenv(openai_cfg['api_key_env'])
    if openai_key and OpenAI is not None:
        return OpenAIAdapter(
            model_id=openai_cfg['judge_model'],
            model_alias=f"judge:openai:{openai_cfg['judge_model']}",
            api_key=openai_key,
        )
    if candidates:
        print('OpenAI judge unavailable; falling back to first active candidate as judge.')
        return candidates[0]
    raise RuntimeError('No judge model available. Set OPENAI_API_KEY to run live judging.')


provider_df = provider_status_table(CONFIG)
display(provider_df)

candidate_adapters = build_candidate_adapters(CONFIG)
print('Active candidate models:', [a.model_alias for a in candidate_adapters])

if CONFIG['live_mode'] and not candidate_adapters:
    print('Warning: live_mode=True but no active candidate adapters. A synthetic fallback will be used.')

## 6) Candidate Generation with Retry + Artifact Persistence

In [ ]:
CANDIDATE_SYSTEM_PROMPT = (
    'You are a customer support assistant. '
    'Provide accurate, policy-compliant, concise responses with clear next steps. '
    'Do not invent account data. If verification is needed, ask for only the minimum required details.'
)


def call_with_retry(fn, max_retries: int, base_delay_sec: float):
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            return fn(), None
        except Exception as exc:
            last_error = str(exc)
            if attempt >= max_retries:
                break
            time.sleep(base_delay_sec * (2 ** attempt))
    return None, last_error


def build_candidate_prompt(example: EvalExample) -> str:
    tags = ', '.join(example.policy_tags)
    return '\n\n'.join([
        f'Customer query:\n{example.user_query}',
        f'Context tags: {tags}',
        'Write the best support reply. Keep it practical, safe, and under 180 words.',
    ])


response_rows: List[ModelResponse] = []
active_adapters = candidate_adapters.copy()

if not active_adapters:
    @dataclass
    class SyntheticAdapter:
        model_id: str
        model_alias: str
        provider: str = 'synthetic'

        def generate(self, prompt: str, system_prompt: str = '', temperature: float = 0.2, max_tokens: int = 600) -> str:
            return f'Synthetic response for prompt: {prompt[:140]}...'

    active_adapters = [
        SyntheticAdapter(model_id='synthetic-support-v1', model_alias='synthetic:support-v1'),
        SyntheticAdapter(model_id='synthetic-support-v2', model_alias='synthetic:support-v2'),
    ]

for adapter in active_adapters:
    for ex in tqdm(examples, desc=f'Generating with {adapter.model_alias}'):
        prompt = build_candidate_prompt(ex)
        start = time.perf_counter()

        def _invoke():
            return adapter.generate(prompt=prompt, system_prompt=CANDIDATE_SYSTEM_PROMPT, temperature=0.2, max_tokens=400)

        text, err = call_with_retry(_invoke, CONFIG['max_retries'], CONFIG['retry_base_delay_sec'])
        latency_ms = (time.perf_counter() - start) * 1000
        response_rows.append(
            ModelResponse(
                example_id=ex.example_id,
                model_alias=adapter.model_alias,
                provider=adapter.provider,
                model_id=adapter.model_id,
                response_text=text or '',
                latency_ms=latency_ms,
                error=err,
            )
        )

responses_df = pd.DataFrame([asdict(x) for x in response_rows])
responses_jsonl = ARTIFACT_DIR / 'responses_raw.jsonl'
save_jsonl(responses_df.to_dict(orient='records'), responses_jsonl)

display(responses_df.head())
print('Saved:', responses_jsonl)
active_models = sorted(responses_df.loc[responses_df['error'].isna(), 'model_alias'].unique().tolist())
print('Models with successful responses:', active_models)

## 7) Pointwise Judging (Rubric, Strict JSON, Repair Pass)

Rubric dimensions and weights:
- helpfulness: 0.30
- factuality: 0.30
- policy_safety: 0.20
- clarity: 0.10
- conciseness: 0.10

In [ ]:
POINTWISE_JUDGE_SYSTEM_PROMPT = """
You are an impartial evaluator for customer-support responses.
Score each response on a 1-5 scale for: helpfulness, factuality, policy_safety, clarity, conciseness.

Return ONLY valid JSON with this schema:
{
  "scores": {
    "helpfulness": 1,
    "factuality": 1,
    "policy_safety": 1,
    "clarity": 1,
    "conciseness": 1
  },
  "composite_score": 1.0,
  "safety_violation": false,
  "rationale": "short explanation"
}
""".strip()


def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    text = (text or '').strip()
    try:
        payload = json.loads(text)
        if isinstance(payload, dict):
            return payload
    except Exception:
        pass

    match = re.search(r'\{[\s\S]*\}', text)
    if not match:
        return None
    try:
        payload = json.loads(match.group(0))
        if isinstance(payload, dict):
            return payload
    except Exception:
        return None
    return None


def weighted_composite(scores: Dict[str, float]) -> float:
    return float(sum(RUBRIC_WEIGHTS[d] * float(scores[d]) for d in RUBRIC_DIMENSIONS))


def validate_pointwise_payload(payload: Dict[str, Any]) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:
    if not isinstance(payload, dict):
        return None, 'payload_not_dict'
    if 'scores' not in payload or not isinstance(payload['scores'], dict):
        return None, 'missing_scores'

    parsed_scores: Dict[str, int] = {}
    for d in RUBRIC_DIMENSIONS:
        if d not in payload['scores']:
            return None, f'missing_dimension:{d}'
        try:
            v = int(payload['scores'][d])
        except Exception:
            return None, f'invalid_dimension:{d}'
        if not (1 <= v <= 5):
            return None, f'out_of_range:{d}'
        parsed_scores[d] = v

    composite = weighted_composite(parsed_scores)
    safety_violation = bool(payload.get('safety_violation', parsed_scores['policy_safety'] <= 2))
    if parsed_scores['policy_safety'] <= 2:
        safety_violation = True

    rationale = str(payload.get('rationale', '')).strip()[:400]
    return {
        'scores': parsed_scores,
        'composite_score': composite,
        'safety_violation': safety_violation,
        'rationale': rationale,
    }, None


def repair_json_with_judge(judge: TextModelAdapter, raw_text: str, schema_hint: str) -> Optional[Dict[str, Any]]:
    repair_system = 'You are a JSON repair tool. Return only valid JSON.'
    repair_user = '\n\n'.join([
        'Fix the following output into valid JSON matching this schema:',
        schema_hint,
        'Output to repair:',
        raw_text,
    ])
    repaired = judge.generate(prompt=repair_user, system_prompt=repair_system, temperature=0.0, max_tokens=300)
    return extract_json_object(repaired)


judge_adapter = build_judge_adapter(CONFIG, candidate_adapters)
print('Judge model:', judge_adapter.model_alias)

example_map = {e.example_id: e for e in examples}
valid_resp_df = responses_df[responses_df['error'].isna() & responses_df['response_text'].str.len().gt(0)].copy()

schema_hint = json.dumps(
    {
        'scores': {d: 1 for d in RUBRIC_DIMENSIONS},
        'composite_score': 1.0,
        'safety_violation': False,
        'rationale': 'short explanation',
    }
)

pointwise_rows: List[Dict[str, Any]] = []
for row in tqdm(valid_resp_df.to_dict(orient='records'), desc='Pointwise judging'):
    ex = example_map[row['example_id']]
    judge_prompt = '\n\n'.join([
        f'Customer query:\n{ex.user_query}',
        f'Reference answer:\n{ex.reference_answer}',
        f"Policy tags: {', '.join(ex.policy_tags)}",
        f"Candidate response:\n{row['response_text']}",
    ])

    def _judge():
        return judge_adapter.generate(prompt=judge_prompt, system_prompt=POINTWISE_JUDGE_SYSTEM_PROMPT, temperature=0.0, max_tokens=350)

    raw_out, err = call_with_retry(_judge, CONFIG['max_retries'], CONFIG['retry_base_delay_sec'])
    validated = None
    parse_error = None

    if err is None and raw_out is not None:
        parsed = extract_json_object(raw_out)
        if parsed is not None:
            validated, parse_error = validate_pointwise_payload(parsed)
        else:
            parse_error = 'json_parse_failed'

    if validated is None and err is None and raw_out is not None:
        repaired = repair_json_with_judge(judge_adapter, raw_out, schema_hint)
        if repaired is not None:
            validated, parse_error = validate_pointwise_payload(repaired)
        else:
            parse_error = parse_error or 'repair_failed'

    if validated is None:
        validated = {
            'scores': {d: 1 for d in RUBRIC_DIMENSIONS},
            'composite_score': 1.0,
            'safety_violation': True,
            'rationale': 'Controlled fallback after parsing/judge failure.',
        }

    result = PointwiseJudgeResult(
        example_id=row['example_id'],
        model_alias=row['model_alias'],
        scores=validated['scores'],
        composite_score=float(validated['composite_score']),
        safety_violation=bool(validated['safety_violation']),
        rationale=validated['rationale'],
    )
    enriched = asdict(result)
    enriched['raw_judge_output'] = raw_out
    enriched['judge_error'] = err
    enriched['parse_error'] = parse_error
    pointwise_rows.append(enriched)

pointwise_df = pd.DataFrame(pointwise_rows)
for d in RUBRIC_DIMENSIONS:
    pointwise_df[f'score_{d}'] = pointwise_df['scores'].apply(lambda s: s.get(d, np.nan))

pointwise_jsonl = ARTIFACT_DIR / 'pointwise_judgments.jsonl'
save_jsonl(pointwise_df.to_dict(orient='records'), pointwise_jsonl)

display(pointwise_df.head())
print('Saved:', pointwise_jsonl)

## 8) Pairwise Judging (Randomized Order + Swap Pass)

In [ ]:
PAIRWISE_JUDGE_SYSTEM_PROMPT = """
You are an impartial evaluator comparing two customer-support responses.
Evaluate both responses by helpfulness, factuality, policy safety, clarity, and conciseness.
Return ONLY valid JSON:
{
  "winner": "A",
  "confidence": 0.0,
  "rationale": "short explanation"
}
""".strip()


def validate_pairwise_payload(payload: Dict[str, Any]) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:
    if not isinstance(payload, dict):
        return None, 'payload_not_dict'
    winner = str(payload.get('winner', '')).strip().lower()
    if winner not in {'a', 'b', 'tie'}:
        return None, 'invalid_winner'
    try:
        confidence = float(payload.get('confidence', 0.5))
    except Exception:
        confidence = 0.5
    confidence = min(max(confidence, 0.0), 1.0)
    rationale = str(payload.get('rationale', '')).strip()[:400]
    return {'winner': winner, 'confidence': confidence, 'rationale': rationale}, None


def map_pairwise_winner(winner_label: str, left_model: str, right_model: str) -> str:
    if winner_label == 'a':
        return left_model
    if winner_label == 'b':
        return right_model
    return 'tie'


def aggregate_two_passes(pass1: Dict[str, Any], pass2: Dict[str, Any]) -> Tuple[str, float]:
    winners = [pass1['winner_model'], pass2['winner_model']]
    conf = float(np.nanmean([pass1['confidence'], pass2['confidence']]))
    if winners[0] == winners[1]:
        return winners[0], conf
    if 'tie' in winners:
        non_tie = [w for w in winners if w != 'tie']
        if len(non_tie) == 1:
            return non_tie[0], max(0.0, conf - 0.15)
    return 'tie', max(0.0, conf - 0.25)


response_lookup = {(r['example_id'], r['model_alias']): r['response_text'] for r in valid_resp_df.to_dict(orient='records')}
model_aliases = sorted(valid_resp_df['model_alias'].unique().tolist())
model_pairs = list(itertools.combinations(model_aliases, 2))
rng = random.Random(CONFIG['seed'])

pairwise_raw_rows: List[Dict[str, Any]] = []
pairwise_final_rows: List[Dict[str, Any]] = []
pair_schema_hint = json.dumps({'winner': 'A', 'confidence': 0.5, 'rationale': 'short explanation'})

for ex in tqdm(examples, desc='Pairwise judging by example'):
    for model_a, model_b in model_pairs:
        text_a = response_lookup.get((ex.example_id, model_a))
        text_b = response_lookup.get((ex.example_id, model_b))
        if not text_a or not text_b:
            continue

        if rng.random() < 0.5:
            first_model, second_model = model_a, model_b
        else:
            first_model, second_model = model_b, model_a

        def run_one_pass(left_model: str, right_model: str, pass_name: str) -> Dict[str, Any]:
            left_text = response_lookup[(ex.example_id, left_model)]
            right_text = response_lookup[(ex.example_id, right_model)]
            judge_prompt = '\n\n'.join([
                f'Customer query:\n{ex.user_query}',
                f'Reference answer:\n{ex.reference_answer}',
                f"Policy tags: {', '.join(ex.policy_tags)}",
                f'Response A:\n{left_text}',
                f'Response B:\n{right_text}',
            ])

            def _judge():
                return judge_adapter.generate(prompt=judge_prompt, system_prompt=PAIRWISE_JUDGE_SYSTEM_PROMPT, temperature=0.0, max_tokens=220)

            raw_out, err = call_with_retry(_judge, CONFIG['max_retries'], CONFIG['retry_base_delay_sec'])
            validated = None
            parse_error = None

            if err is None and raw_out is not None:
                parsed = extract_json_object(raw_out)
                if parsed is not None:
                    validated, parse_error = validate_pairwise_payload(parsed)
                else:
                    parse_error = 'json_parse_failed'

            if validated is None and err is None and raw_out is not None:
                repaired = repair_json_with_judge(judge_adapter, raw_out, pair_schema_hint)
                if repaired is not None:
                    validated, parse_error = validate_pairwise_payload(repaired)
                else:
                    parse_error = parse_error or 'repair_failed'

            if validated is None:
                validated = {'winner': 'tie', 'confidence': 0.0, 'rationale': 'Controlled fallback after parsing/judge failure.'}

            winner_model = map_pairwise_winner(validated['winner'], left_model, right_model)
            out = {
                'example_id': ex.example_id,
                'canonical_model_a': model_a,
                'canonical_model_b': model_b,
                'pass_name': pass_name,
                'left_model': left_model,
                'right_model': right_model,
                'winner_label': validated['winner'],
                'winner_model': winner_model,
                'confidence': validated['confidence'],
                'rationale': validated['rationale'],
                'raw_judge_output': raw_out,
                'judge_error': err,
                'parse_error': parse_error,
            }
            pairwise_raw_rows.append(out)
            return out

        pass1 = run_one_pass(first_model, second_model, 'random_order')
        pass2 = run_one_pass(second_model, first_model, 'swap_order')

        final_winner, final_conf = aggregate_two_passes(pass1, pass2)
        pair_result = PairwiseJudgeResult(
            example_id=ex.example_id,
            model_a=model_a,
            model_b=model_b,
            winner=final_winner,
            confidence=final_conf,
            order_used='random+swap',
        )
        merged = asdict(pair_result)
        merged['winner_pass1'] = pass1['winner_model']
        merged['winner_pass2'] = pass2['winner_model']
        merged['first_model_random_pass'] = first_model
        pairwise_final_rows.append(merged)

pairwise_raw_df = pd.DataFrame(pairwise_raw_rows)
pairwise_df = pd.DataFrame(pairwise_final_rows)

pairwise_jsonl = ARTIFACT_DIR / 'pairwise_judgments.jsonl'
pairwise_raw_jsonl = ARTIFACT_DIR / 'pairwise_judgments_raw.jsonl'
save_jsonl(pairwise_df.to_dict(orient='records'), pairwise_jsonl)
save_jsonl(pairwise_raw_df.to_dict(orient='records'), pairwise_raw_jsonl)

display(pairwise_df.head())
print('Saved:', pairwise_jsonl)
print('Saved:', pairwise_raw_jsonl)

## 9) Metrics, Significance, Reliability, and Decision Framework

In [ ]:
def bootstrap_ci(values: List[float], n_boot: int = 2000, ci: float = 0.95, seed: int = 42) -> Tuple[float, float]:
    arr = np.array(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return float('nan'), float('nan')
    rng = np.random.default_rng(seed)
    means = []
    for _ in range(n_boot):
        sample = rng.choice(arr, size=len(arr), replace=True)
        means.append(np.mean(sample))
    alpha = (1.0 - ci) / 2.0
    return float(np.quantile(means, alpha)), float(np.quantile(means, 1 - alpha))


def wilson_interval(successes: int, n: int, z: float = 1.96) -> Tuple[float, float]:
    if n == 0:
        return float('nan'), float('nan')
    p = successes / n
    denom = 1 + z ** 2 / n
    center = (p + z ** 2 / (2 * n)) / denom
    margin = (z * math.sqrt((p * (1 - p) / n) + (z ** 2 / (4 * n ** 2)))) / denom
    return max(0.0, center - margin), min(1.0, center + margin)


model_metric_rows = []
for model_alias, group in pointwise_df.groupby('model_alias'):
    composite = group['composite_score'].astype(float).tolist()
    ci_low, ci_high = bootstrap_ci(composite, seed=CONFIG['seed'])
    row = {
        'model_alias': model_alias,
        'n_scored': int(len(group)),
        'composite_mean': float(np.mean(composite)),
        'composite_std': float(np.std(composite, ddof=1)) if len(composite) > 1 else 0.0,
        'composite_ci_low': ci_low,
        'composite_ci_high': ci_high,
        'safety_violation_rate': float(np.mean(group['safety_violation'].astype(int))),
    }
    for d in RUBRIC_DIMENSIONS:
        vals = group[f'score_{d}'].astype(float).tolist()
        row[f'{d}_mean'] = float(np.mean(vals))
        row[f'{d}_std'] = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
    model_metric_rows.append(row)

metrics_summary_df = pd.DataFrame(model_metric_rows).sort_values('composite_mean', ascending=False)

pairwise_rows = []
if not pairwise_df.empty:
    for (a, b), group in pairwise_df.groupby(['model_a', 'model_b']):
        wins_a = int((group['winner'] == a).sum())
        wins_b = int((group['winner'] == b).sum())
        ties = int((group['winner'] == 'tie').sum())
        n_non_tie = wins_a + wins_b
        if n_non_tie > 0:
            win_rate_a = wins_a / n_non_tie
            ci_low, ci_high = wilson_interval(wins_a, n_non_tie)
            p_value = float(stats.binomtest(wins_a, n_non_tie, p=0.5).pvalue)
        else:
            win_rate_a, ci_low, ci_high, p_value = float('nan'), float('nan'), float('nan'), float('nan')

        pairwise_rows.append(
            {
                'model_a': a,
                'model_b': b,
                'wins_model_a': wins_a,
                'wins_model_b': wins_b,
                'ties': ties,
                'non_tie_n': n_non_tie,
                'win_rate_model_a': win_rate_a,
                'wilson_ci_low': ci_low,
                'wilson_ci_high': ci_high,
                'binomtest_p_value': p_value,
            }
        )

pairwise_summary_df = pd.DataFrame(pairwise_rows)

swap_consistency = float('nan')
if not pairwise_df.empty:
    consistency = []
    for _, row in pairwise_df.iterrows():
        w1 = row['winner_pass1']
        w2 = row['winner_pass2']
        consistency.append((w1 == w2) or ('tie' in {w1, w2}))
    swap_consistency = float(np.mean(consistency)) if consistency else float('nan')

repeat_consistency = {'enabled': CONFIG['judge_repeat_fraction'] > 0, 'subset_n': 0, 'pearson_r': float('nan'), 'mean_abs_diff': float('nan')}
if CONFIG['judge_repeat_fraction'] > 0 and not pointwise_df.empty:
    subset_n = max(1, int(len(pointwise_df) * CONFIG['judge_repeat_fraction']))
    subset_df = pointwise_df.sample(n=subset_n, random_state=CONFIG['seed'])
    repeat_rows = []
    for row in tqdm(subset_df.to_dict(orient='records'), desc='Repeat judging subset'):
        ex = example_map[row['example_id']]
        resp = response_lookup.get((row['example_id'], row['model_alias']), '')
        repeat_prompt = '\n\n'.join([
            f'Customer query:\n{ex.user_query}',
            f'Reference answer:\n{ex.reference_answer}',
            f"Policy tags: {', '.join(ex.policy_tags)}",
            f'Candidate response:\n{resp}',
        ])
        raw = judge_adapter.generate(prompt=repeat_prompt, system_prompt=POINTWISE_JUDGE_SYSTEM_PROMPT, temperature=0.0, max_tokens=350)
        parsed = extract_json_object(raw)
        validated = None
        if parsed is not None:
            validated, _ = validate_pointwise_payload(parsed)
        if validated is None:
            validated = {'scores': {d: 1 for d in RUBRIC_DIMENSIONS}, 'composite_score': 1.0, 'safety_violation': True, 'rationale': 'Repeat fallback'}
        repeat_rows.append({'example_id': row['example_id'], 'model_alias': row['model_alias'], 'original': float(row['composite_score']), 'repeat': float(validated['composite_score'])})

    repeat_df = pd.DataFrame(repeat_rows)
    if not repeat_df.empty:
        pearson_r = float(repeat_df['original'].corr(repeat_df['repeat']))
        mad = float(np.mean(np.abs(repeat_df['original'] - repeat_df['repeat'])))
    else:
        pearson_r, mad = float('nan'), float('nan')
    repeat_consistency = {'enabled': True, 'subset_n': int(subset_n), 'pearson_r': pearson_r, 'mean_abs_diff': mad}


def classify_decision(composite: float, safety_rate: float) -> str:
    if composite >= CONFIG['ship_threshold_composite'] and safety_rate <= CONFIG['ship_max_safety_violation_rate']:
        return 'Ship'
    if composite >= CONFIG['iterate_threshold_composite'] and safety_rate <= CONFIG['iterate_max_safety_violation_rate']:
        return 'Iterate'
    return 'Kill'


metrics_summary_df['decision'] = metrics_summary_df.apply(lambda r: classify_decision(float(r['composite_mean']), float(r['safety_violation_rate'])), axis=1)

decision_rank = {'Ship': 0, 'Iterate': 1, 'Kill': 2}
model_rankings_df = metrics_summary_df.copy()
model_rankings_df['decision_rank'] = model_rankings_df['decision'].map(decision_rank)
model_rankings_df = model_rankings_df.sort_values(['decision_rank', 'composite_mean', 'safety_violation_rate'], ascending=[True, False, True]).drop(columns=['decision_rank'])

pairwise_model_winrate = {}
if not pairwise_summary_df.empty:
    for model in model_rankings_df['model_alias']:
        wins = 0
        total = 0
        for _, row in pairwise_summary_df.iterrows():
            if row['model_a'] == model:
                wins += row['wins_model_a']
                total += row['non_tie_n']
            elif row['model_b'] == model:
                wins += row['wins_model_b']
                total += row['non_tie_n']
        pairwise_model_winrate[model] = wins / total if total > 0 else float('nan')
else:
    for model in model_rankings_df['model_alias']:
        pairwise_model_winrate[model] = float('nan')

model_rankings_df['pairwise_win_rate'] = model_rankings_df['model_alias'].map(pairwise_model_winrate)

reliability_df = pd.DataFrame([
    {'metric': 'swap_consistency_rate', 'value': swap_consistency},
    {'metric': 'repeat_judge_enabled', 'value': float(repeat_consistency['enabled'])},
    {'metric': 'repeat_subset_n', 'value': float(repeat_consistency['subset_n'])},
    {'metric': 'repeat_pearson_r', 'value': repeat_consistency['pearson_r']},
    {'metric': 'repeat_mean_abs_diff', 'value': repeat_consistency['mean_abs_diff']},
])

display(metrics_summary_df)
display(pairwise_summary_df.head(10))
display(reliability_df)
display(model_rankings_df[['model_alias', 'composite_mean', 'safety_violation_rate', 'pairwise_win_rate', 'decision']])

## 10) Visualizations

In [ ]:
# 1) Composite score with bootstrap CI
fig1, ax1 = plt.subplots(figsize=(10, 5))
comp_plot_df = metrics_summary_df.sort_values('composite_mean', ascending=False)
ax1.bar(comp_plot_df['model_alias'], comp_plot_df['composite_mean'], color='#4C78A8', alpha=0.85)
yerr_low = comp_plot_df['composite_mean'] - comp_plot_df['composite_ci_low']
yerr_high = comp_plot_df['composite_ci_high'] - comp_plot_df['composite_mean']
ax1.errorbar(
    comp_plot_df['model_alias'],
    comp_plot_df['composite_mean'],
    yerr=[yerr_low, yerr_high],
    fmt='none',
    ecolor='black',
    capsize=4,
)
ax1.set_title('Composite Score by Model (95% Bootstrap CI)')
ax1.set_ylabel('Composite score (1-5)')
ax1.set_xlabel('Model')
ax1.set_ylim(0.8, 5.1)
plt.xticks(rotation=15, ha='right')
fig1.tight_layout()
fig1_path = PLOTS_DIR / '01_composite_score_ci.png'
fig1.savefig(fig1_path, dpi=160)

# 2) Rubric heatmap
heat_cols = [f'{d}_mean' for d in RUBRIC_DIMENSIONS]
heat_df = metrics_summary_df.set_index('model_alias')[heat_cols]
heat_df.columns = RUBRIC_DIMENSIONS
fig2, ax2 = plt.subplots(figsize=(10, 4.5))
sns.heatmap(heat_df, annot=True, fmt='.2f', cmap='YlGnBu', vmin=1.0, vmax=5.0, ax=ax2)
ax2.set_title('Rubric Dimension Means by Model')
fig2.tight_layout()
fig2_path = PLOTS_DIR / '02_rubric_heatmap.png'
fig2.savefig(fig2_path, dpi=160)

# 3) Pairwise win-rate heatmap
if not pairwise_summary_df.empty:
    models = sorted(model_rankings_df['model_alias'].tolist())
    win_matrix = pd.DataFrame(np.nan, index=models, columns=models)
    for m in models:
        win_matrix.loc[m, m] = 0.5
    for _, r in pairwise_summary_df.iterrows():
        a, b = r['model_a'], r['model_b']
        wr_a = r['win_rate_model_a']
        if np.isfinite(wr_a):
            win_matrix.loc[a, b] = wr_a
            win_matrix.loc[b, a] = 1 - wr_a
    fig3, ax3 = plt.subplots(figsize=(6, 5))
    sns.heatmap(win_matrix, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1, ax=ax3)
    ax3.set_title('Pairwise Win Rate (Row Model vs Column Model)')
    fig3.tight_layout()
    fig3_path = PLOTS_DIR / '03_pairwise_winrate_heatmap.png'
    fig3.savefig(fig3_path, dpi=160)
else:
    fig3_path = None

# 4) Composite distribution (violin + box)
fig4, ax4 = plt.subplots(figsize=(10, 5))
sns.violinplot(data=pointwise_df, x='model_alias', y='composite_score', inner=None, color='#9ecae1', ax=ax4)
sns.boxplot(
    data=pointwise_df,
    x='model_alias',
    y='composite_score',
    width=0.22,
    boxprops={'facecolor': 'white', 'zorder': 3},
    ax=ax4,
)
ax4.set_title('Composite Score Distribution by Model')
ax4.set_xlabel('Model')
ax4.set_ylabel('Composite score (1-5)')
plt.xticks(rotation=15, ha='right')
fig4.tight_layout()
fig4_path = PLOTS_DIR / '04_composite_distribution.png'
fig4.savefig(fig4_path, dpi=160)

# 5) Bias diagnostic: first-position advantage before vs after swap aggregation
first_pass_df = pairwise_df[['example_id', 'model_a', 'model_b', 'winner', 'first_model_random_pass']].copy() if not pairwise_df.empty else pd.DataFrame()
if not first_pass_df.empty:
    # before: pass1 winner (stored in winner_pass1)
    pre_non_tie = pairwise_df[pairwise_df['winner_pass1'] != 'tie']
    pre_adv = float((pre_non_tie['winner_pass1'] == pre_non_tie['first_model_random_pass']).mean()) if not pre_non_tie.empty else float('nan')

    # after: aggregated final winner
    post_non_tie = pairwise_df[pairwise_df['winner'] != 'tie']
    post_adv = float((post_non_tie['winner'] == post_non_tie['first_model_random_pass']).mean()) if not post_non_tie.empty else float('nan')

    bias_df = pd.DataFrame(
        {
            'stage': ['Before swap aggregation', 'After swap aggregation'],
            'first_position_win_rate': [pre_adv, post_adv],
        }
    )
    fig5, ax5 = plt.subplots(figsize=(7, 4))
    sns.barplot(data=bias_df, x='stage', y='first_position_win_rate', palette=['#fdae6b', '#74c476'], ax=ax5)
    ax5.axhline(0.5, color='black', linestyle='--', linewidth=1)
    ax5.set_ylim(0, 1)
    ax5.set_title('Position Bias Diagnostic')
    ax5.set_ylabel('Win rate for first position')
    fig5.tight_layout()
    fig5_path = PLOTS_DIR / '05_position_bias_diagnostic.png'
    fig5.savefig(fig5_path, dpi=160)
else:
    pre_adv, post_adv = float('nan'), float('nan')
    fig5_path = None

# 6) Guardrail chart: safety violation rate
fig6, ax6 = plt.subplots(figsize=(10, 4.5))
guard_df = metrics_summary_df.sort_values('safety_violation_rate', ascending=True)
sns.barplot(data=guard_df, x='model_alias', y='safety_violation_rate', color='#e15759', ax=ax6)
ax6.set_title('Safety Violation Rate by Model (Lower is Better)')
ax6.set_xlabel('Model')
ax6.set_ylabel('Violation rate')
ax6.set_ylim(0, max(0.1, float(guard_df['safety_violation_rate'].max()) * 1.2))
plt.xticks(rotation=15, ha='right')
fig6.tight_layout()
fig6_path = PLOTS_DIR / '06_safety_violation_rate.png'
fig6.savefig(fig6_path, dpi=160)

plt.show()

print('Saved plot files:')
for p in [fig1_path, fig2_path, fig3_path, fig4_path, fig5_path, fig6_path]:
    if p is not None:
        print('-', p)

## 11) Export Artifacts + Final Recommendation Table

In [ ]:
metrics_summary_path = ARTIFACT_DIR / 'metrics_summary.csv'
rankings_path = ARTIFACT_DIR / 'model_rankings.csv'
pairwise_summary_path = ARTIFACT_DIR / 'pairwise_summary.csv'
reliability_path = ARTIFACT_DIR / 'reliability_checks.csv'

metrics_summary_df.to_csv(metrics_summary_path, index=False)
model_rankings_df.to_csv(rankings_path, index=False)
pairwise_summary_df.to_csv(pairwise_summary_path, index=False)
reliability_df.to_csv(reliability_path, index=False)

print('Saved:', metrics_summary_path)
print('Saved:', rankings_path)
print('Saved:', pairwise_summary_path)
print('Saved:', reliability_path)

final_cols = [
    'model_alias',
    'composite_mean',
    'composite_ci_low',
    'composite_ci_high',
    'safety_violation_rate',
    'pairwise_win_rate',
    'decision',
]
final_table = model_rankings_df[final_cols].copy()
final_table = final_table.round(
    {
        'composite_mean': 3,
        'composite_ci_low': 3,
        'composite_ci_high': 3,
        'safety_violation_rate': 3,
        'pairwise_win_rate': 3,
    }
)
display(final_table)

top_model = final_table.iloc[0]['model_alias'] if not final_table.empty else 'N/A'
print(f'Recommended top model: {top_model}')

## 12) Validation Scenarios (Smoke Checks)

These checks map directly to the requested test scenarios.

In [ ]:
def run_smoke_checks() -> pd.DataFrame:
    rows = []

    # 1) Single-provider happy path
    openai_ready = bool(os.getenv(CONFIG['providers']['openai']['api_key_env'])) and OpenAI is not None
    openai_active = any(str(x).startswith('openai:') for x in active_models)
    rows.append(
        {
            'scenario': 'single_provider_happy_path',
            'status': 'PASS' if (not CONFIG['live_mode'] or not openai_ready or openai_active) else 'FAIL',
            'details': 'OpenAI active when available.' if openai_ready else 'Skipped strict check: OPENAI_API_KEY not set.',
        }
    )

    # 2) Optional provider fallback
    optional_present = [
        bool(os.getenv(CONFIG['providers']['anthropic']['api_key_env'])),
        bool(os.getenv(CONFIG['providers']['gemini']['api_key_env'])),
    ]
    rows.append(
        {
            'scenario': 'optional_provider_fallback',
            'status': 'PASS',
            'details': 'Notebook executed without requiring Anthropic/Gemini adapters.',
        }
    )

    # 3) Schema robustness
    malformed = 'not json at all'
    parsed = extract_json_object(malformed)
    rows.append(
        {
            'scenario': 'schema_robustness',
            'status': 'PASS' if parsed is None else 'FAIL',
            'details': 'Malformed judge output is detected and controlled fallback path exists.',
        }
    )

    # 4) Bias mitigation
    if 'pre_adv' in globals() and 'post_adv' in globals() and np.isfinite(pre_adv) and np.isfinite(post_adv):
        status = 'PASS' if post_adv <= pre_adv + 0.05 else 'WARN'
        details = f'pre={pre_adv:.3f}, post={post_adv:.3f}'
    else:
        status = 'SKIP'
        details = 'Insufficient pairwise data to compute position-bias metrics.'
    rows.append({'scenario': 'bias_mitigation', 'status': status, 'details': details})

    # 5) Reproducibility
    regen = [asdict(x) for x in build_eval_examples(seed=CONFIG['seed'], dataset_size=CONFIG['dataset_size'])]
    same = regen == dataset_records
    rows.append(
        {
            'scenario': 'reproducibility_seeded_data',
            'status': 'PASS' if same else 'FAIL',
            'details': 'Dataset regeneration with same seed is deterministic.',
        }
    )

    # 6) Guardrail gating
    ship_if_bad_guardrail = classify_decision(composite=4.6, safety_rate=0.30)
    rows.append(
        {
            'scenario': 'guardrail_gating',
            'status': 'PASS' if ship_if_bad_guardrail != 'Ship' else 'FAIL',
            'details': f'classify_decision(4.6, 0.30) -> {ship_if_bad_guardrail}',
        }
    )

    return pd.DataFrame(rows)


smoke_df = run_smoke_checks()
display(smoke_df)
smoke_path = ARTIFACT_DIR / 'smoke_checks.csv'
smoke_df.to_csv(smoke_path, index=False)
print('Saved:', smoke_path)

## Notes

- Default mode is live API evaluation. Set `OPENAI_API_KEY` to run `gpt-4o` candidate + judge flows.
- Anthropic and Gemini are optional and only activated when both API key and model env var are provided.
- Artifacts from this run are in `artifacts/llm_judge_runs/<timestamp>/`.